In [ ]:
%%writefile eikonale.h
#pragma once           // il file viene incluso una sola volta
// Risolutore locale 3D (comune ai due metodi) ed errore rispetto alla soluzione esatta della sfera
#include <vector>
#include <cmath>      // fminf, sqrtf
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FIM a nodi sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Errore relativo ||u - U|| / ||U|| rispetto alla soluzione esatta U = distanza dal centro
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h, z = -1.0 + k * h;   // coordinate del nodo (k, i, j)
            double U = std::sqrt(x * x + y * y + z * z);        // soluzione esatta
            double d = u[(k * N + i) * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
            }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}

In [ ]:
%%writefile block_fim.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, sorgente singola, nessun ostacolo
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "eikonale.h"         // INF e solve_quadratic3

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo

// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Elabora un blocco di 8^3 nodi (512 thread, uno per nodo), cuore dell'Alg. 7:
//  1) copia blocco + halo in shared memory;
//  2) niter iterazioni locali su tutti i nodi insieme;
//  3) riscrive in u i propri nodi;
//  4) reduction: convergente se nessun nodo e' cambiato.
// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, int src, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];                // blocco + halo (10^3 valori), nella shared memory
    __shared__ unsigned char s_c[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);            // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * BLOCK + ty) * BLOCK + tx;                                       // numero del thread nel blocco: 0..511
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;                                                  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || (idx == src);             // nodo da non aggiornare: fuori dalla griglia o sorgente (u = 0)
    float f = inside ? F[idx] : 1.0f;                  // F_ijk del nodo

    // 1) Global -> Shared: 1000 celle (blocco + halo), 512 thread
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        int sz = m / ((BLOCK + 2) * (BLOCK + 2));
        int sy = (m / (BLOCK + 2)) % (BLOCK + 2);
        int sx = m % (BLOCK + 2);                             // cella (sz, sy, sx) della shared memory
        int gi = bi * BLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * BLOCK + sx - 1;                         
        int gk = bk * BLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la copia e' completa prima di usarla

    // 2) niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                 // valore prima dell'aggiornamento
        float q = fixed ? p                                  // sorgente / fuori: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // 3) Shared -> Global: ogni thread riscrive il proprio nodo (non l'halo)
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];

    // 4) Cb(b) <- reduction(Cn(b)): AND dei 512 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = BLOCK * BLOCK * BLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

// Step 1 (Alg. 7): ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* list, float* u, const float* F,
                                    int N, int nbx, float h, int src, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u, F, N, nbx, h, src, eps, Cb1);
}

// Inizio di ogni giro: costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco di nodi
    if (t >= n_blocks) return;                               // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A
}

// Step 2, parte 1: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // blockDim = 256
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

// Step 2, parte 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        float* u, const float* F, int N, int nbx, float h,
                                        int src, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u, F, N, nbx, h, src, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}

// Block FIM (Alg. 7) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di giri
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              int src, float eps, int niter, float* ms)
{
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx; // blocchi per lato e blocchi totali
    const size_t bytes = (size_t)N * N * N * sizeof(float); // dimensione in byte della griglia

    std::vector<unsigned char> Cb0(n_blocks, 1);
    const int sk = src / (N * N), si = (src / N) % N, sj = src % N;   // strato, riga e colonna del nodo sorgente
    Cb0[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;   // vale 1 ovunque tranne che nel blocco della sorgente

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));


    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;   // kernel "di servizio": 256 thread, un thread per blocco di nodi
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo (si misura solo il ciclo)
    int iter = 0;
    while (true) {                                           // un giro dell'Alg. 7
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_active, d_list, d_count, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (count == 0) break;  // se count == 0, lista A vuota, termina ciclo
        ++iter;
        // Step 1: foreach b in A in parallelo
        update_active_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_u, d_F, N, nbx, h, src, niter, eps, d_Cb1);    // si lanciano count blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // Step 2: solo i vicini dei blocchi appena convergenti (i candidati)
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        build_candidate_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                          d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                    d_u, d_F, N, nbx, h, src, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1));
    CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

In [ ]:
%%writefile node_fim.cuh
#pragma once           // il file viene incluso una sola volta
// FIM a nodi (Alg. 6 della tesi) sulla GPU: lista attiva di nodi, un thread per nodo, niente shared memory
#include "block_fim.cuh"   // CUDA_CHECK, INF e solve_quadratic3

// Nuovo u del nodo x, leggendo i 6 vicini dalla memoria globale (INF fuori dalla griglia)
__device__ float solve_global(const float* u, const float* F, int x, int N, float h)
{
    int k = x / (N * N), i = (x / N) % N, j = x % N;             // strato, riga e colonna del nodo
    float a = fminf(j > 0 ? u[x - 1] : INF, j < N - 1 ? u[x + 1] : INF);              // vicini lungo x
    float b = fminf(i > 0 ? u[x - N] : INF, i < N - 1 ? u[x + N] : INF);              // vicini lungo y
    float c = fminf(k > 0 ? u[x - N * N] : INF, k < N - 1 ? u[x + N * N] : INF);      // vicini lungo z
    return fminf(u[x], solve_quadratic3(a, b, c, F[x], h));      // return min(u_x, u)
}

// Alg. 6, prima parte: ogni nodo x della lista A si aggiorna (p <- u(x), q <- g(x), u(x) <- q)
__global__ void node_update(const int* list, int count, float* u, const float* F,
                            int N, float h, float eps, unsigned char* conv)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;               // un thread per nodo della lista
    if (t >= count) return;
    int x = list[t];
    float p = u[x];
    float q = solve_global(u, F, x, N, h);
    u[x] = q;
    conv[t] = fabsf(p - q) < eps;                                // |p - q| < eps: nodo convergente
}

// Alg. 6, seconda parte: un nodo non convergente resta in A; uno convergente esce
// e prova i vicini fuori da A: se un vicino migliora, entra nella nuova lista
__global__ void node_neighbors(const int* list, int count, const unsigned char* conv,
                               const unsigned char* in_list, float* u, const float* F, int N, float h,
                               int* next_flag, int* next_list, int* next_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t >= count) return;
    int x = list[t];
    if (!conv[t]) {                                              // non convergente: resta attivo
        if (atomicExch(&next_flag[x], 1) == 0) next_list[atomicAdd(next_count, 1)] = x;
        return;
    }
    int k = x / (N * N), i = (x / N) % N, j = x % N;
    int  nb[6] = { x - 1,  x + 1,     x - N, x + N,     x - N * N, x + N * N };   // sinistra, destra, sotto, sopra, davanti, dietro
    bool ok[6] = { j > 0, j < N - 1, i > 0, i < N - 1, k > 0,     k < N - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d] || in_list[nb[d]]) continue;                  // fuori griglia, oppure gia' in A
        int y = nb[d];
        float p = u[y];
        float q = solve_global(u, F, y, N, h);
        if (q < p) {                                             // p > q: il vicino migliora
            u[y] = q;
            if (atomicExch(&next_flag[y], 1) == 0)               // aggiungi a A (una volta sola)
                next_list[atomicAdd(next_count, 1)] = y;
        }
    }
}

// Aggiornamento dei flag di appartenenza: la lista vecchia esce...
__global__ void clear_in_list(const int* list, int count, unsigned char* in_list)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) in_list[list[t]] = 0;
}

// ...e la nuova entra (next_flag torna a 0 per l'iterazione successiva)
__global__ void set_in_list(const int* list, int count, unsigned char* in_list, int* next_flag)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) { in_list[list[t]] = 1; next_flag[list[t]] = 0; }
}

// FIM a nodi (Alg. 6) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di iterazioni
int node_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
             int src, float eps, float* ms)
{
    const int n = N * N * N;                                     // numero di nodi
    const size_t bytes = (size_t)n * sizeof(float);
    const int threads = 256;

    // Memoria sulla GPU
    float *d_u, *d_F; int *d_list, *d_next, *d_count, *d_flag; unsigned char *d_conv, *d_in;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_list, n * sizeof(int)));            // lista A attuale
    CUDA_CHECK(cudaMalloc(&d_next, n * sizeof(int)));            // lista A del passo successivo
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_flag, n * sizeof(int)));            // 1 = gia' messo nella nuova lista
    CUDA_CHECK(cudaMalloc(&d_conv, n));                          // convergenza dei nodi della lista
    CUDA_CHECK(cudaMalloc(&d_in, n));                            // 1 = nodo in A
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_flag, 0, n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_in, 0, n));

    // All'inizio A contiene solo la sorgente
    int count = 1;
    CUDA_CHECK(cudaMemcpy(d_list, &src, sizeof(int), cudaMemcpyHostToDevice));
    set_in_list<<<1, threads>>>(d_list, count, d_in, d_flag);
    CUDA_CHECK(cudaGetLastError());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                          // si misura solo il ciclo
    int iter = 0;
    while (count > 0) {                                          // while A != vuoto
        ++iter;
        const int blocks = (count + threads - 1) / threads;
        node_update<<<blocks, threads>>>(d_list, count, d_u, d_F, N, h, eps, d_conv);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        node_neighbors<<<blocks, threads>>>(d_list, count, d_conv, d_in, d_u, d_F, N, h,
                                            d_flag, d_next, d_count);
        CUDA_CHECK(cudaGetLastError());
        clear_in_list<<<blocks, threads>>>(d_list, count, d_in);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));   // dimensione della nuova A
        if (count > 0) {
            set_in_list<<<(count + threads - 1) / threads, threads>>>(d_next, count, d_in, d_flag);
            CUDA_CHECK(cudaGetLastError());
        }
        std::swap(d_list, d_next);                               // la nuova lista diventa quella attuale
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_next)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_flag)); CUDA_CHECK(cudaFree(d_conv)); CUDA_CHECK(cudaFree(d_in));
    return iter;
}

In [ ]:
%%writefile main.cu
// Confronto FIM a nodi (Alg. 6) e Block FIM (Alg. 7), entrambi sulla GPU, nel caso della sfera:
// F = 1, nessun ostacolo, sorgente al centro (soluzione esatta = distanza dal centro)
#include <cstdio>
#include <algorithm>       // std::min
#include <fstream>
#include "eikonale.h"      // risolutore locale ed errore rispetto alla soluzione esatta
#include "block_fim.cuh"   // Block FIM (Alg. 7)
#include "node_fim.cuh"    // FIM a nodi (Alg. 6)

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d\n", prop.name, prop.multiProcessorCount);

    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza (uguale per i due metodi)
    const int niter = 12;                                    // iterazioni locali del Block FIM (prove: 12, 14)
    printf("speedup = t FIM a nodi / t Block FIM\n\n");
    printf("    N   iter. FIM   giri Block FIM   max|FIM - BlockFIM|   errore FIM   errore Block FIM   t FIM [ms]   t Block FIM [ms]   speedup\n");
    std::ofstream tab("confronto.txt");                      // una riga per N: N, iterazioni, giri, tempi (per il grafico)
    for (int Nk : { 33, 65, 129, 193, 257, 385 }) {
        const int ripetizioni = 3;                           // ogni tempo = il minimo di 3 misure (solo metodi GPU: costano poco)
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo [-1, 1]^3
        const int n = Nk * Nk * Nk;
        const int srck = ((Nk / 2) * Nk + Nk / 2) * Nk + (Nk / 2);   // sorgente al centro
        const std::vector<float> Fk(n, 1.0f);                // F = 1: u = distanza dal centro
        std::vector<float> u0(n, INF);                       // condizione iniziale: 0 nella sorgente, INF altrove
        u0[srck] = 0.0f;

        std::vector<float> u_node, u_block;
        double t_node = 1e30, t_block = 1e30;
        int it_node = 0, it_block = 0;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu' veloce
            float ms = 0.0f;
            u_node = u0;
            it_node = node_fim(u_node, Fk, Nk, hk, srck, eps, &ms);              // GPU: Alg. 6
            t_node = std::min(t_node, (double)ms);

            u_block = u0;
            it_block = block_fim(u_block, Fk, Nk, hk, srck, eps, niter, &ms);    // GPU: Alg. 7
            t_block = std::min(t_block, (double)ms);
        }

        float d_nb = 0.0f;                                   // i due metodi devono dare lo stesso u
        for (int q = 0; q < n; ++q) d_nb = fmaxf(d_nb, fabsf(u_node[q] - u_block[q]));
        printf("%5d   %9d   %14d   %19.2e   %10.3e   %16.3e   %10.2f   %16.2f   %7.1f\n", Nk, it_node, it_block,
               d_nb, relative_error(u_node, Nk, hk), relative_error(u_block, Nk, hk), t_node, t_block, t_node / t_block);
        // Utile solo per la visualizzazione su Python
        tab << Nk << " " << it_node << " " << it_block << " " << t_node << " " << t_block << "\n";
        if (Nk == 257) {                                     // i due campi u per la vista 3D (Python)
            std::ofstream a("u_fim_257.bin", std::ios::binary), b("u_block_257.bin", std::ios::binary);
            a.write(reinterpret_cast<const char*>(u_node.data()), u_node.size() * sizeof(float));
            b.write(reinterpret_cast<const char*>(u_block.data()), u_block.size() * sizeof(float));
        }
    }
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 main.cu -o confronto && ./confronto

In [ ]:

# Grafico del confronto: tempi, speedup e iterazioni globali al variare di N (dati scritti dal programma C++)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FILE = "confronto.txt"                                              # una riga per N: N, iterazioni FIM, giri Block FIM, t FIM, t Block FIM
N, it_fim, giri, t_fim, t_block = np.loadtxt(FILE, unpack=True)
ARANCIO, BLU, GRIGIO = "#eb6834", "#2a78d6", "#52514e"              # FIM a nodi, Block FIM, speedup

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
                    subplot_titles=("Tempo [ms], scala logaritmica", "Speedup = t FIM / t Block FIM", "Iterazioni globali"))
for y, nome, colore, col in [(t_fim, "FIM a nodi (Alg. 6)", ARANCIO, 1), (t_block, "Block FIM (Alg. 7)", BLU, 1),
                             (it_fim, "FIM a nodi (Alg. 6)", ARANCIO, 3), (giri, "Block FIM (Alg. 7)", BLU, 3)]:
    fig.add_trace(go.Scatter(x=N, y=y, mode="lines+markers", name=nome, legendgroup=nome, showlegend=(col == 1),
                             line=dict(color=colore, width=2), marker=dict(size=8)), row=1, col=col)
fig.add_trace(go.Scatter(x=N, y=t_fim / t_block, mode="lines+markers+text", text=[f"{s:.1f}" for s in t_fim / t_block],
                         textposition="top center", line=dict(color=GRIGIO, width=2), marker=dict(size=8),
                         name="speedup", showlegend=False), row=1, col=2)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_xaxes(title_text="N (nodi per lato)", tickvals=N)
fig.update_layout(template="plotly_white", height=460, margin=dict(l=40, r=20, t=60, b=40),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=1.18))
fig.write_html("confronto_tempi.html", include_plotlyjs="cdn")     # anche come file da scaricare
fig.show()


In [ ]:
# Vista 3D: il fronte u = t del FIM a nodi (sinistra) e del Block FIM (destra), N = 257, F = 1, sorgente al centro.
# Lo slider (o Play) fa avanzare t: i due fronti devono essere uguali (stessa soluzione).
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from skimage.measure import marching_cubes

N = 65                                                              # griglia del grafico: un nodo ogni 4 della soluzione a N = 257
h = 2.0 / (N - 1)
u_fim = np.fromfile("u_fim_257.bin", dtype=np.float32).reshape(257, 257, 257)[::4, ::4, ::4]       # u[k, i, j] = u(z, y, x)
u_block = np.fromfile("u_block_257.bin", dtype=np.float32).reshape(257, 257, 257)[::4, ::4, ::4]
print(f"max |u FIM - u Block FIM| = {np.abs(u_fim - u_block).max():.2e}")
istanti = np.round(np.linspace(0.1, 0.9, 9), 2)
ARANCIO, BLU = "#eb6834", "#2a78d6"

def fronte(u, t, colore, nome, scena):                              # superficie u = t (fronte all'istante t) nel riquadro "scena"
    v, f, _, _ = marching_cubes(u, level=t, spacing=(h, h, h))
    p = v - 1.0                                                     # vertici in [-1, 1], ordine (z, y, x)
    return go.Mesh3d(x=p[:, 2], y=p[:, 1], z=p[:, 0], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                     color=colore, opacity=0.85, flatshading=True, name=nome, scene=scena)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=("FIM a nodi (Alg. 6)", "Block FIM (Alg. 7)"))
fig.add_trace(fronte(u_fim, istanti[0], ARANCIO, "FIM a nodi", "scene"))
fig.add_trace(fronte(u_block, istanti[0], BLU, "Block FIM", "scene2"))
fig.frames = [go.Frame(data=[fronte(u_fim, t, ARANCIO, "FIM a nodi", "scene"), fronte(u_block, t, BLU, "Block FIM", "scene2")],
                       traces=[0, 1], name=f"{t:.2f}") for t in istanti]
cubo = dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube")
fig.update_layout(
    scene=cubo, scene2=cubo, height=600, margin=dict(l=0, r=0, t=40, b=80), showlegend=False,
    updatemenus=[dict(type="buttons", direction="left", x=0.1, y=0, xanchor="right", yanchor="top", pad=dict(r=10, t=70), buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.1, len=0.9, y=0, yanchor="top", currentvalue=dict(prefix="fronte u = t = "), pad=dict(t=50),
                  steps=[dict(label=f"{t:.2f}", method="animate",
                              args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for t in istanti])])
fig.write_html("confronto_fronti.html", include_plotlyjs="cdn")
fig.show()